

Phase 2a: β条件付き DDPM for 32×32 Polyakov Maps
==================================================

┌──────────────────────────────────────────────────────────────┐
│ メタ認知的自己監査: 設計判断の全体像                          │
│                                                              │
│ 判断 1: 入力表現 → 2ch (Re, Im)                             │
│   根拠: θ は ±π で不連続。(Re,Im) は滑らかだが拘束あり。   │
│   監査: Phase 3 で |Re²+Im²| の偏差を測定。                │
│                                                              │
│ 判断 2: ノイズスケジュール → cosine                          │
│   根拠: 小画像では cosine が linear より安定                 │
│   (Nichol & Dhariwal 2021)。                                │
│   監査: loss curve の挙動で判定。plateau なら linear に変更。│
│                                                              │
│ 判断 3: β注入 → time embedding に加算                       │
│   根拠: AdaGN は time+β を同一ベクトルで処理。              │
│   シンプルで、β と t の相互作用を自然に学べる。             │
│   監査: held-out β での生成品質。                            │
│                                                              │
│ 判断 4: CircularPad2d → 全 Conv で PBC 準拠                 │
│   根拠: failure analysis の指針4。物理的境界条件と整合。     │
│   監査: 生成画像の境界 vs 中心の統計量比較。                │
│                                                              │
│ 判断 5: アーキテクチャ → 3レベル U-Net (32→16→8→4)         │
│   根拠: 32×32 には十分。4×4 bottleneck で大域情報を捕捉。  │
│   ~7.5M params。T4 で batch=64, 1-2hr 学習。               │
│   監査: 生成 ⟨|P|⟩ の MC との一致。                         │
└──────────────────────────────────────────────────────────────┘

## Colabでの実行メモ
- このノートブックは添付された `Phase2a_DDPM.py` をセル実行向けに分割したものです。
- `DATA_DIR` は **Phase 1 の `.npz` データが入ったフォルダ** を指すようにしてください。
- Colab では `Runtime` → `Change runtime type` から GPU を有効化して実行してください。
- Google Drive 上のデータを使う場合は、下の任意セルで Drive をマウントして `DATA_DIR` を変更してください。


  ZIPの解凍セル

In [ ]:
import os

# まず中身を確認
BASE = '/kaggle/input/datasets/mokafe/phase1-dataset'
print("中身:", os.listdir(BASE)[:10])

# サブフォルダがある場合に備えて探索
for root, dirs, files in os.walk(BASE):
    npz = [f for f in files if f.endswith('.npz')]
    if npz:
        DATA_DIR = root
        print(f"\n.npz 発見: {DATA_DIR}")
        print(f"ファイル数: {len(npz)}")
        print(f"サンプル: {npz[:5]}")
        break
else:
    # ZIPが未解凍で残っている場合 → /kaggle/working/ に解凍
    import zipfile, glob
    zips = glob.glob(os.path.join(BASE, '*.zip'))
    if zips:
        DATA_DIR = '/kaggle/working/data/'
        os.makedirs(DATA_DIR, exist_ok=True)
        with zipfile.ZipFile(zips[0], 'r') as zf:
            zf.extractall(DATA_DIR)
        print(f"解凍完了 → {DATA_DIR}")
        print(os.listdir(DATA_DIR)[:10])
    else:
        print("エラー: .npz も .zip も見つかりません")

In [ ]:
# 1. セットアップ
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import math
import os
import json
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


In [ ]:
# 2. CircularPad2d + Conv2d wrapper
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# ┌──────────────────────────────────────────────────────────┐
# │ 判断4の実装: PBC 準拠の畳み込み                          │
# │                                                          │
# │ PyTorch の Conv2d は padding_mode='circular' を           │
# │ サポートしている。全ての Conv2d でこれを使う。           │
# │                                                          │
# │ 監査: padding_mode='circular' は kernel_size=3 の場合     │
# │ 各辺1ピクセルだけ周期的に折り返す。                      │
# │ 32×32 格子で kernel=3 なら物理的に正しい。               │
# │ ただし GroupNorm のパディングは Conv の外なので影響なし。 │
# └──────────────────────────────────────────────────────────┘

def circ_conv(in_ch, out_ch, kernel_size=3, stride=1, bias=True):
    """PBC 準拠 Conv2d。kernel_size=3, padding=1, circular。"""
    return nn.Conv2d(in_ch, out_ch, kernel_size, stride=stride,
                     padding=kernel_size // 2, padding_mode='circular',
                     bias=bias)


In [ ]:
# 3. Time + β Embedding
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class SinusoidalEmbedding(nn.Module):
    """Sinusoidal positional embedding for diffusion timestep."""
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
        args = t[:, None].float() * freqs[None, :]
        return torch.cat([args.sin(), args.cos()], dim=-1)


class ConditionEmbedding(nn.Module):
    """
    Time + β → 共通 embedding ベクトル。

    ┌──────────────────────────────────────────────────────┐
    │ 判断3の実装: β を time embedding に加算               │
    │                                                      │
    │ time_emb: sinusoidal(t) → MLP → d_model              │
    │ beta_emb: MLP(β_normed) → d_model                   │
    │ combined: time_emb + beta_emb → MLP → d_model        │
    │                                                      │
    │ 加算（vs. 連結）の理由:                               │
    │ - 連結は次元を倍にし、下流の全 AdaGN の入力次元が変わる│
    │ - 加算は「β は t と同じ空間に住む条件」という仮定     │
    │ - 格子ゲージ理論では β は連続的に変化する結合定数で、  │
    │   t と同様にスカラー条件 → 加算が自然                 │
    │                                                      │
    │ 監査: held-out β (1.75, 1.85) での生成品質で検証。    │
    │ 補間が悪ければ連結に切り替え or cross-attention。      │
    └──────────────────────────────────────────────────────┘
    """
    def __init__(self, d_model=256):
        super().__init__()
        self.time_sin = SinusoidalEmbedding(d_model)
        self.time_mlp = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.SiLU(),
            nn.Linear(d_model, d_model),
        )
        self.beta_mlp = nn.Sequential(
            nn.Linear(1, d_model),
            nn.SiLU(),
            nn.Linear(d_model, d_model),
        )
        self.combine = nn.Sequential(
            nn.SiLU(),
            nn.Linear(d_model, d_model),
        )

    def forward(self, t, beta_normed):
        """
        t: (B,) integer timestep
        beta_normed: (B,) float in [0,1]
        """
        t_emb = self.time_mlp(self.time_sin(t))
        b_emb = self.beta_mlp(beta_normed[:, None])
        return self.combine(t_emb + b_emb)


In [ ]:
# 4. ResBlock with Adaptive Group Normalization
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class AdaGN(nn.Module):
    """Adaptive Group Normalization: condition → (scale, shift)."""
    def __init__(self, n_channels, d_cond, n_groups=8):
        super().__init__()
        self.gn = nn.GroupNorm(n_groups, n_channels, affine=False)
        self.proj = nn.Linear(d_cond, 2 * n_channels)

    def forward(self, x, cond):
        """x: (B,C,H,W), cond: (B, d_cond)"""
        h = self.gn(x)
        scale, shift = self.proj(cond)[:, :, None, None].chunk(2, dim=1)
        return h * (1 + scale) + shift


class ResBlock(nn.Module):
    """
    ResNet block with circular padding and AdaGN conditioning.

    conv1 → AdaGN(cond) → SiLU → conv2 → + skip
    """
    def __init__(self, in_ch, out_ch, d_cond=256, n_groups=8):
        super().__init__()
        self.conv1 = circ_conv(in_ch, out_ch)
        self.adagn = AdaGN(out_ch, d_cond, n_groups=min(n_groups, out_ch))
        self.conv2 = circ_conv(out_ch, out_ch)
        self.act = nn.SiLU()

        if in_ch != out_ch:
            self.skip_proj = nn.Conv2d(in_ch, out_ch, 1)
        else:
            self.skip_proj = nn.Identity()

    def forward(self, x, cond):
        h = self.act(self.conv1(x))
        h = self.act(self.adagn(h, cond))
        h = self.conv2(h)
        return h + self.skip_proj(x)


In [ ]:
# 5. Self-Attention (bottleneck only)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class SelfAttention(nn.Module):
    """
    Multi-head self-attention for spatial features.

    ┌──────────────────────────────────────────────────────┐
    │ 4×4 bottleneck にのみ配置（16 tokens → O(256) 計算） │
    │ 8×8 以上では計算コストが急増するので使わない。       │
    │                                                      │
    │ これは Locality Trap への直接対策:                    │
    │ bottleneck の Self-Attention が大域的な                │
    │ Polyakov ドメイン構造の一貫性を保証する。             │
    └──────────────────────────────────────────────────────┘
    """
    def __init__(self, channels, n_heads=4):
        super().__init__()
        self.n_heads = n_heads
        self.norm = nn.GroupNorm(8, channels)
        self.qkv = nn.Conv2d(channels, 3 * channels, 1)
        self.out = nn.Conv2d(channels, channels, 1)

    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x)
        qkv = self.qkv(h).reshape(B, 3, self.n_heads, C // self.n_heads, H * W)
        q, k, v = qkv[:, 0], qkv[:, 1], qkv[:, 2]

        # 修正1：bhem -> bhdm に変更（QとKの内積をとるため）
        attn = torch.einsum("bhdn,bhdm->bhnm", q, k) / math.sqrt(C // self.n_heads)
        attn = attn.softmax(dim=-1)

        # 修正2：bhem -> bhdm に変更（エラーが出ていた箇所）
        out = torch.einsum("bhnm,bhdm->bhdn", attn, v)

        out = out.reshape(B, C, H, W)
        return x + self.out(out)


In [ ]:
# 6. U-Net
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class UNet(nn.Module):
    """
    3-level U-Net for 32×32 Polyakov maps.

    Encoder:    32×32×64 → 16×16×128 → 8×8×256
    Bottleneck: 4×4×256 (with self-attention)
    Decoder:    8×8×256 → 16×16×128 → 32×32×64

    ┌──────────────────────────────────────────────────────┐
    │ 判断5の実装: チャンネル数の根拠                       │
    │                                                      │
    │ 32×32 画像には (64,128,256) で十分。                  │
    │ CIFAR-10 DDPM (Ho et al. 2020) は (128,256,256,256)  │
    │ だが、あれは10クラス×5万枚。我々は1タスク×4万枚。    │
    │ 過学習リスクを考えると小さいモデルの方が安全。        │
    │                                                      │
    │ 監査: train loss vs validation loss の乖離で判定。    │
    │ 過学習 → dropout 追加 or チャンネル数削減。           │
    │ 未学習 → チャンネル数を (128,256,512) に増やす。     │
    └──────────────────────────────────────────────────────┘
    """
    def __init__(self, in_ch=2, ch_mults=(64, 128, 256), d_cond=256):
        super().__init__()
        self.cond_emb = ConditionEmbedding(d_cond)
        c0, c1, c2 = ch_mults

        # Input projection
        self.in_conv = circ_conv(in_ch, c0)

        # Encoder
        self.down1a = ResBlock(c0, c0, d_cond)
        self.down1b = ResBlock(c0, c0, d_cond)
        self.pool1 = nn.Conv2d(c0, c0, 3, stride=2, padding=1, padding_mode='circular')

        self.down2a = ResBlock(c0, c1, d_cond)
        self.down2b = ResBlock(c1, c1, d_cond)
        self.pool2 = nn.Conv2d(c1, c1, 3, stride=2, padding=1, padding_mode='circular')

        self.down3a = ResBlock(c1, c2, d_cond)
        self.down3b = ResBlock(c2, c2, d_cond)
        self.pool3 = nn.Conv2d(c2, c2, 3, stride=2, padding=1, padding_mode='circular')

        # Bottleneck (4×4)
        self.bot1 = ResBlock(c2, c2, d_cond)
        self.bot_attn = SelfAttention(c2)
        self.bot2 = ResBlock(c2, c2, d_cond)

        # Decoder
        self.up3 = nn.ConvTranspose2d(c2, c2, 2, stride=2)
        self.dec3a = ResBlock(c2 + c2, c2, d_cond)  # skip connection
        self.dec3b = ResBlock(c2, c2, d_cond)

        self.up2 = nn.ConvTranspose2d(c2, c1, 2, stride=2)
        self.dec2a = ResBlock(c1 + c1, c1, d_cond)
        self.dec2b = ResBlock(c1, c1, d_cond)

        self.up1 = nn.ConvTranspose2d(c1, c0, 2, stride=2)
        self.dec1a = ResBlock(c0 + c0, c0, d_cond)
        self.dec1b = ResBlock(c0, c0, d_cond)

        # Output
        self.out_norm = nn.GroupNorm(8, c0)
        self.out_conv = circ_conv(c0, in_ch)
        nn.init.zeros_(self.out_conv.weight)
        nn.init.zeros_(self.out_conv.bias)

    def forward(self, x, t, beta_normed):
        """
        x: (B, in_ch, 32, 32)  noisy input
        t: (B,)                 integer timestep
        beta_normed: (B,)       normalized β ∈ [0,1]
        Returns: predicted noise ε (same shape as x)
        """
        cond = self.cond_emb(t, beta_normed)

        # Encode
        h = self.in_conv(x)
        s1 = self.down1b(self.down1a(h, cond), cond)
        h = self.pool1(s1)
        s2 = self.down2b(self.down2a(h, cond), cond)
        h = self.pool2(s2)
        s3 = self.down3b(self.down3a(h, cond), cond)
        h = self.pool3(s3)

        # Bottleneck
        h = self.bot1(h, cond)
        h = self.bot_attn(h)
        h = self.bot2(h, cond)

        # Decode
        h = self.up3(h)
        h = self.dec3b(self.dec3a(torch.cat([h, s3], dim=1), cond), cond)
        h = self.up2(h)
        h = self.dec2b(self.dec2a(torch.cat([h, s2], dim=1), cond), cond)
        h = self.up1(h)
        h = self.dec1b(self.dec1a(torch.cat([h, s1], dim=1), cond), cond)

        return self.out_conv(F.silu(self.out_norm(h)))


In [ ]:
# 7. Diffusion schedule
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class CosineSchedule:
    """
    Cosine noise schedule (Nichol & Dhariwal 2021).

    ┌──────────────────────────────────────────────────────┐
    │ 判断2の実装: cosine vs linear                        │
    │                                                      │
    │ Linear schedule は大画像では安定だが、32×32 では      │
    │ 初期ステップでノイズが強すぎて信号が早期に消失する。 │
    │ Cosine は小さな t で信号をより長く保持する。          │
    │                                                      │
    │ s=0.008 は Nichol & Dhariwal の推奨値。               │
    │ T=1000 は標準。小画像なら T=500 でも十分な可能性あり。│
    │                                                      │
    │ 監査: T=500 と T=1000 で生成品質を比較。              │
    │ 差がなければ T=500 に下げて推論速度を2倍にする。     │
    └──────────────────────────────────────────────────────┘
    """
    def __init__(self, T=1000, s=0.008):
        self.T = T
        steps = torch.arange(T + 1, dtype=torch.float64)
        f = torch.cos((steps / T + s) / (1 + s) * math.pi / 2) ** 2
        alpha_bar = f / f[0]
        # Clip betas to prevent numerical issues
        betas = 1 - alpha_bar[1:] / alpha_bar[:-1]
        betas = betas.clamp(max=0.999)

        self.betas = betas.float()
        self.alphas = (1 - self.betas).float()
        self.alpha_bar = torch.cumprod(self.alphas, dim=0).float()
        self.sqrt_alpha_bar = torch.sqrt(self.alpha_bar)
        self.sqrt_one_minus_alpha_bar = torch.sqrt(1 - self.alpha_bar)

    def to(self, device):
        self.betas = self.betas.to(device)
        self.alphas = self.alphas.to(device)
        self.alpha_bar = self.alpha_bar.to(device)
        self.sqrt_alpha_bar = self.sqrt_alpha_bar.to(device)
        self.sqrt_one_minus_alpha_bar = self.sqrt_one_minus_alpha_bar.to(device)
        return self

    def q_sample(self, x0, t, noise=None):
        """Forward process: x_t = √ᾱ_t x_0 + √(1-ᾱ_t) ε"""
        if noise is None:
            noise = torch.randn_like(x0)
        sqrt_ab = self.sqrt_alpha_bar[t][:, None, None, None]
        sqrt_omab = self.sqrt_one_minus_alpha_bar[t][:, None, None, None]
        return sqrt_ab * x0 + sqrt_omab * noise, noise

    def p_sample(self, model, x_t, t, beta_normed):
        """Reverse process: one denoising step."""
        B = x_t.shape[0]
        t_tensor = torch.full((B,), t, device=x_t.device, dtype=torch.long)

        with torch.no_grad():
            eps_pred = model(x_t, t_tensor, beta_normed)

        alpha = self.alphas[t]
        alpha_bar = self.alpha_bar[t]
        beta = self.betas[t]

        # Mean
        coef1 = 1 / torch.sqrt(alpha)
        coef2 = beta / self.sqrt_one_minus_alpha_bar[t]
        mean = coef1 * (x_t - coef2 * eps_pred)

        if t > 0:
            sigma = torch.sqrt(beta)
            noise = torch.randn_like(x_t)
            return mean + sigma * noise
        else:
            return mean

    @torch.no_grad()
    def sample(self, model, shape, beta_normed, show_progress=True):
        """Full reverse process: x_T → x_0"""
        x = torch.randn(shape, device=beta_normed.device)
        steps = range(self.T - 1, -1, -1)
        if show_progress:
            try:
                from tqdm import tqdm
                steps = tqdm(steps, desc="Sampling")
            except ImportError:
                pass
        for t in steps:
            x = self.p_sample(model, x, t, beta_normed)
        return x


In [ ]:
# 8. Dataset
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class PolyakovDataset(Dataset):
    """
    Phase 1 の .npz ファイルから PyTorch Dataset を構築。

    D4 augmentation を on-the-fly で適用（メモリ効率）。
    """
    def __init__(self, data_dir, channels="re_im", augment_d4=True):
        self.channels = channels
        self.augment = augment_d4

        files = sorted([
            os.path.join(data_dir, f) for f in os.listdir(data_dir)
            if f.endswith('.npz')
        ])

        raw_images = []
        raw_betas = []

        for fpath in files:
            data = np.load(fpath)
            beta = float(os.path.basename(fpath).split('beta_')[1].split('.npz')[0])
            n = data['poly_re'].shape[0]

            for i in range(n):
                if channels == "re_im":
                    img = np.stack([data['poly_re'][i], data['poly_im'][i]])
                elif channels == "re":
                    img = data['poly_re'][i:i+1]
                elif channels == "phase":
                    img = data['poly_phase'][i:i+1] / np.pi  # normalize to [-1, 1]
                raw_images.append(img)
                raw_betas.append(beta)

        self.raw_images = np.array(raw_images, dtype=np.float32)
        self.raw_betas = np.array(raw_betas, dtype=np.float32)
        self.n_raw = len(self.raw_images)

        # β normalization
        self.beta_min = self.raw_betas.min()
        self.beta_max = self.raw_betas.max()

        n_aug = 8 if augment_d4 else 1
        print(f"Dataset: {self.n_raw} raw × {n_aug} augment = {self.n_raw * n_aug} total")
        print(f"  channels={channels}, shape={self.raw_images[0].shape}")
        print(f"  β ∈ [{self.beta_min:.2f}, {self.beta_max:.2f}]")

    def __len__(self):
        return self.n_raw * (8 if self.augment else 1)

    def __getitem__(self, idx):
        if self.augment:
            raw_idx = idx // 8
            aug_idx = idx % 8
        else:
            raw_idx = idx
            aug_idx = 0

        img = self.raw_images[raw_idx].copy()  # (C, H, W)
        beta = self.raw_betas[raw_idx]

        # D4 augmentation on spatial dims
        rot = aug_idx % 4
        flip = aug_idx // 4
        if rot > 0:
            img = np.rot90(img, rot, axes=(-2, -1)).copy()
        if flip:
            img = np.flip(img, axis=-1).copy()

        beta_normed = (beta - self.beta_min) / (self.beta_max - self.beta_min + 1e-8)

        return torch.from_numpy(img), torch.tensor(beta_normed, dtype=torch.float32)

    def beta_to_normed(self, beta_phys):
        return (beta_phys - self.beta_min) / (self.beta_max - self.beta_min + 1e-8)


In [ ]:
# 9. Training loop
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def train(model, schedule, dataset, epochs=200, batch_size=64,
          lr=2e-4, ema_decay=0.9999, save_dir="phase2_checkpoints",
          log_every=50, save_every=50):
    """
    DDPM training with EMA.

    ┌──────────────────────────────────────────────────────┐
    │ EMA (Exponential Moving Average):                    │
    │ 生成品質を安定させるための標準テクニック。            │
    │ decay=0.9999 は ~10K steps で1/e 減衰。              │
    │ 125K steps → 十分に平滑化される。                    │
    │                                                      │
    │ 監査: EMA weight と raw weight の loss 比較。         │
    │ EMA が大幅に良ければ、学習率が高すぎるかも。         │
    └──────────────────────────────────────────────────────┘
    """
    os.makedirs(save_dir, exist_ok=True)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                       num_workers=0, pin_memory=(device.type == 'cuda'), drop_last=True)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    # EMA
    ema_model = UNet(in_ch=model.in_conv.weight.shape[1]).to(device)
    ema_model.load_state_dict(model.state_dict())
    ema_model.eval()

    # Cosine LR schedule with warmup
    total_steps = epochs * len(loader)
    warmup_steps = min(1000, total_steps // 10)

    def lr_lambda(step):
        if step < warmup_steps:
            return step / warmup_steps
        progress = (step - warmup_steps) / (total_steps - warmup_steps)
        return 0.5 * (1 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    loss_history = []
    global_step = 0
    t_start = time.time()

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        n_batches = 0

        for images, beta_normed in loader:
            images = images.to(device)
            beta_normed = beta_normed.to(device)

            # Sample random timesteps
            t = torch.randint(0, schedule.T, (images.shape[0],), device=device)

            # Forward diffusion
            x_noisy, noise = schedule.q_sample(images, t)

            # Predict noise
            noise_pred = model(x_noisy, t, beta_normed)
            loss = F.mse_loss(noise_pred, noise)

            optimizer.zero_grad()
            loss.backward()
            # Gradient clipping for stability
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            # EMA update
            with torch.no_grad():
                for p_ema, p_model in zip(ema_model.parameters(), model.parameters()):
                    p_ema.data.mul_(ema_decay).add_(p_model.data, alpha=1 - ema_decay)

            epoch_loss += loss.item()
            n_batches += 1
            global_step += 1

            if global_step % log_every == 0:
                avg = epoch_loss / n_batches
                lr_now = scheduler.get_last_lr()[0]
                elapsed = time.time() - t_start
                steps_per_sec = global_step / elapsed
                eta_min = (total_steps - global_step) / steps_per_sec / 60
                print(f"  step {global_step:6d}/{total_steps} | "
                      f"loss={avg:.5f} | lr={lr_now:.2e} | "
                      f"ETA={eta_min:.0f}min")

        avg_loss = epoch_loss / n_batches
        loss_history.append(avg_loss)

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs} | loss={avg_loss:.5f}")

        if (epoch + 1) % save_every == 0:
            ckpt = {
                "epoch": epoch + 1,
                "model": model.state_dict(),
                "ema_model": ema_model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "loss_history": loss_history,
                "global_step": global_step,
            }
            ckpt_path = os.path.join(save_dir, f"ckpt_epoch{epoch+1:04d}.pt")
            torch.save(ckpt, ckpt_path)
            print(f"  Checkpoint saved: {ckpt_path}")

    # Save final
    final_path = os.path.join(save_dir, "ckpt_final.pt")
    torch.save({
        "epoch": epochs,
        "model": model.state_dict(),
        "ema_model": ema_model.state_dict(),
        "loss_history": loss_history,
        "global_step": global_step,
        "dataset_info": {
            "beta_min": float(dataset.beta_min),
            "beta_max": float(dataset.beta_max),
            "channels": dataset.channels,
            "n_raw": dataset.n_raw,
        },
    }, final_path)
    print(f"\nFinal checkpoint: {final_path}")

    return loss_history, ema_model


In [ ]:
# 10. 生成 + 物理的検証
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

@torch.no_grad()
def generate_and_evaluate(ema_model, schedule, dataset,
                          betas_phys, n_per_beta=100):
    """
    各 β で n_per_beta 枚生成し、⟨|P|⟩ を計算。
    MC ground truth と比較。

    ┌──────────────────────────────────────────────────────┐
    │ 判断1の監査: |Re²+Im²| 拘束チェック                  │
    │ 各サイトで Re² + Im² を計算し、1.0 からの偏差を報告。│
    │ これが大きければ、生成データは非物理的。              │
    └──────────────────────────────────────────────────────┘
    """
    ema_model.eval()
    results = {}

    for beta_phys in betas_phys:
        beta_normed = dataset.beta_to_normed(beta_phys)
        beta_tensor = torch.full((n_per_beta,), beta_normed, device=device)

        # Generate
        samples = schedule.sample(
            ema_model,
            (n_per_beta, 2, 32, 32),  # 2ch: Re, Im
            beta_tensor,
            show_progress=False
        )  # (N, 2, 32, 32)

        re = samples[:, 0].cpu().numpy()  # (N, 32, 32)
        im = samples[:, 1].cpu().numpy()

        # ⟨|P|⟩ per sample
        avg_re = re.mean(axis=(1, 2))  # (N,)
        avg_im = im.mean(axis=(1, 2))
        poly_abs = np.sqrt(avg_re**2 + avg_im**2)

        # |P(x,y)|² constraint check
        site_norm_sq = re**2 + im**2  # (N, 32, 32)
        norm_deviation = np.abs(site_norm_sq - 1.0).mean()

        results[beta_phys] = {
            "poly_mean": float(poly_abs.mean()),
            "poly_std": float(poly_abs.std()),
            "norm_deviation": float(norm_deviation),
        }

        print(f"β={beta_phys:5.2f}: ⟨|P|⟩={poly_abs.mean():.4f}±{poly_abs.std():.4f}, "
              f"|Re²+Im²-1|={norm_deviation:.4f}")

    return results


def plot_results(loss_history, gen_results, mc_reference=None):
    """学習曲線 + 生成 vs MC 比較プロット。"""
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Loss curve
    axes[0].plot(loss_history, lw=0.8)
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("MSE Loss")
    axes[0].set_title("Training loss")
    axes[0].set_yscale("log")

    # ⟨|P|⟩: generated vs MC
    betas = sorted(gen_results.keys())
    gen_poly = [gen_results[b]["poly_mean"] for b in betas]
    gen_std = [gen_results[b]["poly_std"] for b in betas]

    axes[1].errorbar(betas, gen_poly, yerr=gen_std, fmt='o-',
                    capsize=4, label='DDPM generated')
    if mc_reference:
        mc_betas = sorted(mc_reference.keys())
        mc_poly = [mc_reference[b][0] for b in mc_betas]
        mc_err = [mc_reference[b][1] for b in mc_betas]
        axes[1].errorbar(mc_betas, mc_poly, yerr=mc_err, fmt='s--',
                        capsize=4, label='MC ground truth')
    axes[1].set_xlabel("β")
    axes[1].set_ylabel("⟨|P|⟩")
    axes[1].set_title("Generated vs MC")
    axes[1].legend()

    # Norm deviation
    norm_dev = [gen_results[b]["norm_deviation"] for b in betas]
    axes[2].bar(range(len(betas)), norm_dev, tick_label=[f"{b:.1f}" for b in betas])
    axes[2].set_xlabel("β")
    axes[2].set_ylabel("|Re²+Im²-1|")
    axes[2].set_title("Unit circle constraint violation")
    axes[2].axhline(0.1, ls='--', color='red', alpha=0.5, label='threshold')
    axes[2].legend()

    plt.tight_layout()
    plt.savefig("phase2_results.png", dpi=150, bbox_inches='tight')
    plt.show()


## 実行セル

In [ ]:

CHANNELS = "re_im"                  # 2ch: (Re P, Im P)
EPOCHS = 200
BATCH_SIZE = 64
LR = 2e-4
T_DIFFUSION = 1000


In [ ]:
# Dataset
dataset = PolyakovDataset(DATA_DIR, channels=CHANNELS, augment_d4=True)

# Model
in_ch = 2 if CHANNELS == "re_im" else 1
model = UNet(in_ch=in_ch).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"\nModel: {n_params:,} parameters ({n_params/1e6:.1f}M)")

# Schedule
schedule = CosineSchedule(T=T_DIFFUSION).to(device)


In [ ]:
print(f"\n{'='*60}")
print(f"Training: {EPOCHS} epochs, batch={BATCH_SIZE}, lr={LR}")
print(f"{'='*60}\n")

loss_history, ema_model = train(
    model, schedule, dataset,
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR,
    save_dir="/kaggle/working/phase2_checkpoints",
    log_every=100, save_every=50,
)


In [ ]:
print(f"\n{'='*60}")
print("Generating samples for evaluation")
print(f"{'='*60}\n")

# 学習に使ったβ + held-out β
eval_betas = [0.5, 1.0, 1.5, 1.7, 1.9, 2.0, 2.5, 4.0, 6.0, 10.0]
held_out = [1.75, 1.85, 3.0, 8.0]  # 学習に含まない β
all_eval = sorted(set(eval_betas + held_out))

gen_results = generate_and_evaluate(ema_model, schedule, dataset,
                                    all_eval, n_per_beta=100)

# MC reference (Phase 0/1 values)
mc_ref = {
    0.5: (0.029, 0.002), 1.0: (0.032, 0.002), 1.5: (0.063, 0.002),
    1.7: (0.114, 0.005), 1.9: (0.262, 0.020), 2.0: (0.378, 0.025),
    2.5: (0.562, 0.004), 4.0: (0.715, 0.004), 6.0: (0.805, 0.003),
    10.0: (0.883, 0.002),
}

# Pass/fail
print(f"\n{'='*60}")
print("Pass/fail check: generated ⟨|P|⟩ vs MC")
print(f"{'='*60}")
for beta in sorted(gen_results.keys()):
    gen_p = gen_results[beta]["poly_mean"]
    if beta in mc_ref:
        mc_p, mc_e = mc_ref[beta]
        diff = abs(gen_p - mc_p)
        tag = "TRAIN" if beta in eval_betas else "HELD-OUT"
        status = "✓" if diff < 3 * mc_e + 0.05 else "✗"
        print(f"  β={beta:5.2f} [{tag:>8}]: gen={gen_p:.3f}, mc={mc_p:.3f}, "
              f"Δ={diff:.3f} {status}")
    else:
        print(f"  β={beta:5.2f} [HELD-OUT]: gen={gen_p:.3f} (no MC ref)")

plot_results(loss_history, gen_results, mc_ref)
print("\n完了。phase2_results.png を確認してください。")
